# Bronze Source Landscape

## Clinical Trial Intelligence Platform

### Objective

This notebook establishes the source landscape for the Bronze layer of the Clinical Trial Intelligence Platform before production ingestion.

The analysis is used to:

- identify all datasets delivered to the Amazon S3 landing zone,
- verify source-path accessibility,
- document source file naming and delivery patterns,
- understand file arrival cadence,
- and establish a complete inventory of datasets expected by the Bronze layer.

This notebook focuses only on the physical source landscape and delivery characteristics. Detailed per-feed schema and source profiling are performed in the corresponding source exploration notebooks, while ingestion strategy decisions are documented separately in the Bronze ingestion design notebook.

## 1. Source Landing Zone Configuration

### Question

What source systems and dataset families are delivered to the Amazon S3 landing zone?

### Purpose

Before evaluating individual files, the landing-zone structure is inspected to establish the physical organization of the source data available for Bronze ingestion.

This check identifies the top-level source families and confirms that the expected clinical data domains are accessible from Databricks.

In [0]:
landing_root = "s3://clinical-trial-intelligence-platform-sk/Landing/"
landing_items = dbutils.fs.ls(landing_root)
display(
    spark.createDataFrame(
        [   (
                item.name.rstrip("/"),
                item.path,
                "Directory" if item.isDir() else "File"
            )
            for item in landing_items
        ],
        ["source_family", "source_path", "object_type"]
    ).orderBy("source_family")
)

### Result

The Amazon S3 landing zone contains **seven top-level source families**:

- `CTMS`
- `EDC`
- `Lab`
- `Safety`
- `master`
- `protocol`
- `reference`

All seven source families are accessible from the Databricks environment and are organized as separate directories under the common `Landing/` prefix.

### Conclusion

The landing-zone connectivity check confirms that the expected clinical operational, protocol, master, and reference source families are available for Bronze-layer ingestion.

At this stage, no ingestion strategy is assigned. The next step is to inspect the datasets contained within each source family and establish a complete Bronze source inventory.

## 2. Bronze Dataset Inventory

### Question

What datasets are physically available within each source family in the S3 landing zone?

### Purpose

A complete dataset inventory is required to verify that every source expected by the Bronze layer is represented in the landing zone.

This check expands each top-level source family and records the immediate dataset-level directories or files available beneath it. The resulting inventory provides the source-level evidence for mapping landed datasets to Bronze tables in later design steps.

In [0]:
dataset_inventory = []
for family in landing_items:
    if family.isDir():
        for item in dbutils.fs.ls(family.path):
            dataset_inventory.append(
                (
                    family.name.rstrip("/"),
                    item.name.rstrip("/"),
                    item.path,
                    "Directory" if item.isDir() else "File"
                )
            )
dataset_inventory_df = spark.createDataFrame(
    dataset_inventory,
    [
        "source_family",
        "dataset",
        "source_path",
        "object_type"
    ]
)
display(
    dataset_inventory_df.orderBy(
        "source_family",
        "dataset"
    )
)

### Dataset Inventory Summary

The dataset-level inventory is summarized by source family to verify how many landed datasets or source objects are present under each top-level source domain.

In [0]:
from pyspark.sql import functions as F
inventory_summary_df = (
    dataset_inventory_df
    .groupBy("source_family")
    .agg(
        F.count("*").alias("dataset_count")
    )
    .orderBy("source_family")
)
display(inventory_summary_df)

In [0]:
total_datasets = dataset_inventory_df.count()
print(f"Total dataset-level objects discovered: {total_datasets}")

In [0]:
display(dataset_inventory_df.select(
        "source_family",
        "dataset",
        "object_type"
    )
    .orderBy(
        "source_family",
        "dataset"
    )
)

## 3. Source File Counts

### Question

How many source files are currently available for each dataset-level object in the landing zone?

### Purpose

File counts provide a first view of source delivery behaviour across the Bronze landing zone.

The comparison helps distinguish datasets delivered through multiple files over time from datasets represented by individual source files, without assigning an ingestion strategy at this stage.

In [0]:
file_count_results = []

for row in dataset_inventory_df.collect():

    source_family = row["source_family"]
    dataset = row["dataset"]
    source_path = row["source_path"]
    object_type = row["object_type"]

    if object_type == "Directory":
        files = [
            item
            for item in dbutils.fs.ls(source_path)
            if not item.isDir()
        ]
        file_count = len(files)

    else:
        # The dataset-level object itself is already a file
        file_count = 1

    file_count_results.append(
        (
            source_family,
            dataset,
            file_count
        )
    )

file_count_df = spark.createDataFrame(
    file_count_results,
    [
        "source_family",
        "dataset",
        "file_count"
    ]
)

display(
    file_count_df.orderBy(
        "source_family",
        "dataset"
    )
)

### Result

The file inventory shows two distinct physical delivery patterns in the landing zone.

The operational clinical feeds contain repeated file deliveries:

- `subjects` — 11 files
- `visits` — 10 files
- `lab_results` — 10 files
- `adverse_events` — 10 files

The CTMS, master, protocol, and reference source objects currently contain or represent a single source file each.

An additional `EDC.txt` object is present under the EDC source family. It is recorded as part of the physical landing-zone inventory but is not treated as a clinical dataset at this stage.

### Conclusion

The landing zone contains both recurring multi-file feeds and single-file source objects. This establishes that source delivery behaviour is not uniform across the platform.

The observed file-count differences are recorded here as source-landscape evidence only. The corresponding streaming or materialized ingestion decisions are documented separately in the Bronze ingestion design notebook.

## 4. Source File Naming Patterns

### Question

What naming conventions are used for files delivered to the S3 landing zone?

### Purpose

Source file names are inspected to determine whether deliveries follow consistent and identifiable naming conventions.

This check records whether recurring feeds contain date-versioned filenames and whether single-file sources use stable names. Detailed use of filename-derived metadata for ingestion or downstream sequencing is documented in the corresponding source exploration and ingestion design notebooks.

In [0]:
file_name_results = []

for row in dataset_inventory_df.collect():

    source_family = row["source_family"]
    dataset = row["dataset"]
    source_path = row["source_path"]
    object_type = row["object_type"]

    if object_type == "Directory":

        for item in dbutils.fs.ls(source_path):

            if not item.isDir():
                file_name_results.append(
                    (
                        source_family,
                        dataset,
                        item.name
                    )
                )

    else:

        file_name_results.append(
            (
                source_family,
                dataset,
                dataset
            )
        )

file_names_df = spark.createDataFrame(
    file_name_results,
    [
        "source_family",
        "dataset",
        "file_name"
    ]
)

display(
    file_names_df.orderBy(
        "source_family",
        "dataset",
        "file_name"
    )
)

### Result

The landing-zone files follow identifiable naming conventions.

Recurring clinical feeds use **date-versioned filenames**, where the delivery date is embedded in the file name. Examples include:

- `subjects_YYYYMMDD.csv`
- `visits_YYYYMMDD.csv`
- `lab_results_YYYYMMDD.csv`
- `adverse_events_YYYYMMDD.csv`

Other source objects also use identifiable dataset-specific names. Master and protocol datasets include dated filenames, while reference datasets use stable descriptive filenames such as `country_region.csv`, `geography.csv`, and `unit_mapping.csv`.

### Conclusion

The landing zone uses consistent dataset-oriented naming conventions. Recurring operational deliveries can be distinguished by their date-versioned filenames, while reference files are identifiable through stable descriptive names.

These naming characteristics are recorded as source-landscape evidence. Their use for ingestion sequencing and lineage is evaluated in the corresponding source exploration and Bronze ingestion design notebooks.

## 5. Source Delivery Cadence

### Question

What delivery cadence is observable across the landed source files?

### Purpose

File modification timestamps are inspected to understand how frequently source files arrive in the landing zone.

This provides physical evidence of recurring versus less-frequent deliveries while keeping ingestion-strategy decisions outside the scope of this notebook.

In [0]:
delivery_results = []

for row in dataset_inventory_df.collect():

    source_family = row["source_family"]
    dataset = row["dataset"]
    source_path = row["source_path"]
    object_type = row["object_type"]

    if object_type == "Directory":

        for item in dbutils.fs.ls(source_path):

            if not item.isDir():
                delivery_results.append(
                    (
                        source_family,
                        dataset,
                        item.name,
                        item.modificationTime
                    )
                )

    else:
        # Direct file object
        matching_item = [
            item
            for item in dbutils.fs.ls(
                source_path.rsplit("/", 1)[0] + "/"
            )
            if item.path == source_path
        ]

        if matching_item:
            delivery_results.append(
                (
                    source_family,
                    dataset,
                    dataset,
                    matching_item[0].modificationTime
                )
            )

delivery_cadence_df = (
    spark.createDataFrame(
        delivery_results,
        [
            "source_family",
            "dataset",
            "file_name",
            "modification_time_ms"
        ]
    )
    .withColumn(
        "modification_timestamp",
        (F.col("modification_time_ms") / 1000)
        .cast("timestamp")
    )
)

display(
    delivery_cadence_df
    .select(
        "source_family",
        "dataset",
        "file_name",
        "modification_timestamp"
    )
    .orderBy(
        "source_family",
        "dataset",
        "modification_timestamp"
    )
)

### Result

The landing-zone inventory contains multiple date-versioned deliveries for the recurring clinical feeds. The filenames represent different logical delivery dates, while several S3 modification timestamps are clustered around the same physical upload period.

More recent Subject files also show later modification timestamps, demonstrating that additional files can be added to the landing path over time.

### Conclusion

The source files provide evidence of repeated deliveries over time. However, S3 modification timestamps represent the physical object-write time and do not necessarily represent the original source delivery date.

Therefore, the date embedded in the source filename is retained as the logical delivery indicator, while file modification time remains useful as technical lineage metadata. Detailed sequencing logic is evaluated in the feed-specific exploration and Bronze ingestion design notebooks.

## 6. Source Accessibility Validation

### Question

Are all identified source-family paths accessible from the Databricks environment?

### Purpose

The source-family paths are validated to confirm that the Bronze exploration can enumerate the expected landing-zone locations without access failures.

This is a technical accessibility check only; source-content validation is performed in the corresponding feed-specific exploration notebooks.

In [0]:
accessibility_results = []

for item in landing_items:
    try:
        dbutils.fs.ls(item.path)

        accessibility_results.append(
            (
                item.name.rstrip("/"),
                item.path,
                "Accessible"
            )
        )

    except Exception:
        accessibility_results.append(
            (
                item.name.rstrip("/"),
                item.path,
                "Not Accessible"
            )
        )

accessibility_df = spark.createDataFrame(
    accessibility_results,
    [
        "source_family",
        "source_path",
        "access_status"
    ]
)

display(
    accessibility_df.orderBy("source_family")
)

### Result

All seven identified source-family paths — `CTMS`, `EDC`, `Lab`, `Safety`, `master`, `protocol`, and `reference` — are accessible from the Databricks environment.

No source-family path returned an accessibility failure during the landing-zone inspection.

### Conclusion

The accessibility validation confirms that the complete identified source landscape can be enumerated from Databricks.

Detailed content, schema, and file-level behaviour are evaluated separately in the corresponding source exploration notebooks.

## 7. Landscape Summary

The Bronze source-landscape exploration established the physical structure and delivery characteristics of the Amazon S3 landing zone before detailed feed-level analysis.

### Key Findings

- **7 source families** were identified: `CTMS`, `EDC`, `Lab`, `Safety`, `master`, `protocol`, and `reference`.
- **18 dataset-level source objects** were discovered across these source families.
- The landing zone currently contains **55 physical source files**.
- The recurring clinical feeds contain multiple deliveries:
  - `subjects` — 11 files
  - `visits` — 10 files
  - `lab_results` — 10 files
  - `adverse_events` — 10 files
- CTMS, master, protocol, and reference source objects currently contain or represent single-file deliveries.
- Recurring clinical feeds use date-versioned filenames, while reference datasets use stable descriptive filenames.
- S3 modification timestamps provide technical object-write metadata, while dates embedded in source filenames provide a clearer logical delivery indicator for the observed recurring feeds.
- All **7 source-family paths** were successfully accessed from Databricks.

### Conclusion

The source landscape contains both recurring multi-file clinical feeds and single-file source objects, with identifiable naming and delivery characteristics across the landing zone.

These findings establish the physical source baseline required for detailed Bronze exploration. Feed-level schema behaviour, file consistency, grain, key completeness, and source-format characteristics are evaluated in the corresponding source exploration notebooks.

The final ingestion strategy, including streaming versus materialized processing, schema management, evolution behaviour, and lineage requirements, is documented separately in `06_bronze_ingestion_design`.